In [28]:
from parsers.lr.lr1 import LR1Parser
from parsers.lr.lr_parsing_engine import LREngine

from parsers.grammars import bison_to_ruleset

In [29]:
grammar = [
    "P -> L",
    "S -> id := id",
    "S -> if id then S",
    "S -> if id then S else S",
    "L -> S",
    "L -> L ; S",
]
bison = r"""
%token id assign then if else sem

%start P

%%

    P
        : L ;
    S 
        : id assign id
        | if id then S
        | if id then S else S
        ;
    L
        : S 
        | L sem S ;
"""


In [30]:
p = LR1Parser(bison_to_ruleset(bison)).to_lalr()

2026-06-14 11:42:04.762 | DEBUG    | parsers.base_parser:__init__:34 - RuleSet(start_rule_idx=0, rules={0: RuleType(lhs='$accept', rhs=('P', '$end')), 1: RuleType(lhs='P', rhs=('L',)), 2: RuleType(lhs='S', rhs=('id', 'assign', 'id')), 3: RuleType(lhs='S', rhs=('if', 'id', 'then', 'S')), 4: RuleType(lhs='S', rhs=('if', 'id', 'then', 'S', 'else', 'S')), 5: RuleType(lhs='L', rhs=('S',)), 6: RuleType(lhs='L', rhs=('L', 'sem', 'S'))}, eol='$end')


In [31]:
p.bison_like_report()

State 8 conflict: s/r


Grammar

  0000 $accept -> P $end
  0001 P -> L
  0002 S -> id assign id
  0003 S -> if id then S
  0004 S -> if id then S else S
  0005 L -> S
  0006 L -> L sem S


Terminals, with rules where they appear

$end: 0
assign: 2
else: 4
id: 2 3 4
if: 3 4
sem: 6
then: 3 4


Nonterminals, with rules where they appear

S
  on left: 2 3 4, on right: 3 4 5 6
L
  on left: 5 6, on right: 1 6
$accept
  on left: 0
P
  on left: 1, on right: 0


---=== States ===---

state 0
    S -> if id then S else . S
    S -> . if id then S
    S -> . if id then S else S
    S -> . id assign id

    S   go to state 12
    id  shift and go to state 21
    if  shift and go to state 14


state 1
    L -> L sem S .

    $end  reduce using rule 6 (L)
    sem   reduce using rule 6 (L)


state 2
    S -> id assign . id

    id  shift and go to state 7


state 3
    S -> if id . then S else S
    S -> if id . then S

    then  shift and go to state 10


state 6
    S -> . if id then S
    $accept

In [32]:
len(p.states)

15

In [33]:
print(p.to_tabulate())

┌────┬──────────┬──────┬──────┬────────┬────────┬────────┬───────┬─────┬─────┬───────────┬─────┐
│    │ assign   │ id   │ if   │ $end   │ then   │ else   │ sem   │ S   │ L   │ $accept   │ P   │
├────┼──────────┼──────┼──────┼────────┼────────┼────────┼───────┼─────┼─────┼───────────┼─────┤
│  0 │          │ s21  │ s14  │        │        │        │       │ g12 │     │           │     │
├────┼──────────┼──────┼──────┼────────┼────────┼────────┼───────┼─────┼─────┼───────────┼─────┤
│  1 │          │      │      │ r6     │        │        │ r6    │     │     │           │     │
├────┼──────────┼──────┼──────┼────────┼────────┼────────┼───────┼─────┼─────┼───────────┼─────┤
│  2 │          │ s7   │      │        │        │        │       │     │     │           │     │
├────┼──────────┼──────┼──────┼────────┼────────┼────────┼───────┼─────┼─────┼───────────┼─────┤
│  3 │          │      │      │        │ s10    │        │       │     │     │           │     │
├────┼──────────┼──────┼──────

In [34]:
conflicts = p.get_conflicts()
state, sym, _ = conflicts[0]

cell = list(p.parsing_table[state][sym])
p.parsing_table[state][sym] = set(cell[1:])

In [35]:
print(p.to_tabulate())

┌────┬──────────┬──────┬──────┬────────┬────────┬────────┬───────┬─────┬─────┬───────────┬─────┐
│    │ assign   │ id   │ if   │ $end   │ then   │ else   │ sem   │ S   │ L   │ $accept   │ P   │
├────┼──────────┼──────┼──────┼────────┼────────┼────────┼───────┼─────┼─────┼───────────┼─────┤
│  0 │          │ s21  │ s14  │        │        │        │       │ g12 │     │           │     │
├────┼──────────┼──────┼──────┼────────┼────────┼────────┼───────┼─────┼─────┼───────────┼─────┤
│  1 │          │      │      │ r6     │        │        │ r6    │     │     │           │     │
├────┼──────────┼──────┼──────┼────────┼────────┼────────┼───────┼─────┼─────┼───────────┼─────┤
│  2 │          │ s7   │      │        │        │        │       │     │     │           │     │
├────┼──────────┼──────┼──────┼────────┼────────┼────────┼───────┼─────┼─────┼───────────┼─────┤
│  3 │          │      │      │        │ s10    │        │       │     │     │           │     │
├────┼──────────┼──────┼──────

In [36]:
e = LREngine(p)

In [37]:
code_sample = """
if id then     
    if id then
        id assign id
    else
        id assign id sem
        id assign id $end
""".split()

In [38]:
def print_iter(stack, states, symbol, action):
    for el in stack:
        print_el = next(iter(el.keys())) if isinstance(el, dict) else el
        print(print_el, end=", ")
    print(f" {symbol=}, {states[-1]=}, {action=}")

In [39]:
import json

In [40]:
out = e.parse(code_sample, iteration_callback=print_iter)

 symbol='if', states[-1]=6, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=14)
if,  symbol='id', states[-1]=14, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=3)
if, id,  symbol='then', states[-1]=3, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=10)
if, id, then,  symbol='if', states[-1]=10, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=14)
if, id, then, if,  symbol='id', states[-1]=14, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=3)
if, id, then, if, id,  symbol='then', states[-1]=3, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=10)
if, id, then, if, id, then,  symbol='id', states[-1]=10, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=21)
if, id, then, if, id, then, id,  symbol='assign', states[-1]=21, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=2)
if, id, then, if, id, then, id, assign,  symbol='id', states[-1]=2, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=7)
if, id, then, if, id, then, id, assign, id,  symbol='else', states[-1

In [41]:
from parsers.lr.helpers import pretify_stack

In [42]:
print(pretify_stack(out))

P
  L
    L
      S
        S
          id
            id
            assign
        sem
        L
          S
            S
              S
                S
                  S
                    S
                      id
                        id
                        assign
                    else
                    then
                    id
                    if
                then
                id
                if



In [43]:
print(json.dumps(out, indent=2))

{
  "P": {
    "L": {
      "L": {
        "S": {
          "S": {
            "id": "id",
            "assign": "assign"
          }
        },
        "sem": "sem",
        "L": {
          "L": {
            "S": {
              "S": {
                "S": {
                  "S": {
                    "S": {
                      "S": {
                        "id": "id",
                        "assign": "assign"
                      }
                    },
                    "else": "else",
                    "then": "then",
                    "id": "id",
                    "if": "if"
                  }
                },
                "then": "then",
                "id": "id",
                "if": "if"
              }
            }
          }
        }
      }
    }
  }
}
